In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM


In [6]:
tokenizer = AutoTokenizer.from_pretrained("andresnowak/Qwen3-0.6B-instruction-finetuned")
model = AutoModelForCausalLM.from_pretrained("andresnowak/Qwen3-0.6B-instruction-finetuned")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-0.8B")

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [2]:
model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [2]:
def extract_relation(model, tokenizer, sentence, subject, object_):
    prompt = f"""
    Extract relations of type Located_In, Work_For, OrgBased_In, Live_In, Kill, or No_Relation. 
    Example 1: Sentence = 'The director of BigCorp is Alison Potatosson.', Subject = 'Alison Potatosson', Object = 'BigCorp', Relation = 'Work_For'."
    Example 2: Sentence = 'London is busy today. Mary walks in the park.', Subject = 'London', Object = 'Mary', Relation = 'No_Relation'."
    Example 3: Sentence = 'Bristol nightclub Motion has moved to a new location.', Subject = 'Motion', Object = 'Bristol', Relation = 'OrgBased_In'."
    Example 4: Sentence = {sentence}, Subject = {subject}, Object = {object_}, Relation = """

    inputs = tokenizer(prompt, return_tensors="pt")
    output_ids = model.generate(**inputs, max_new_tokens=3)
    return tokenizer.decode(output_ids[0][-3:], skip_special_tokens=True)

In [3]:
sentence = "Satya Nadella is the Chairman and CEO of Microsoft, a position he has held since taking over from Steve Ballmer in 2014. "
subject = "Satya Nadella"
object_ = "Microsoft"
pred = extract_relation(model, tokenizer, sentence, subject, object_)
print(f"Sentence: {sentence}")
print(f"Predicted relation: {pred}")


[transformers] Both `max_new_tokens` (=3) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentence: Satya Nadella is the Chairman and CEO of Microsoft, a position he has held since taking over from Steve Ballmer in 2014. 
Predicted relation:  'Work_For


In [45]:
def classify_room(model, tokenizer, objects):
    prompt = f"""
    Infer room type based on objects in it. The room will always be one of the following: Kitchen, Bathroom, Living_Room, or Bedroom. 
    Example 1: Objects = 'Microwave, ChoppingBoard, apple.', Room = 'Kitchen'."
    Example 2: Objects = 'Toilet, Shower, Toothbrush.', Room = 'Bathroom'."
    Example 3: Objects = 'Sofa, TV, Bookshelf.', Room = 'Living_Room'."
    Example 4: Objects = 'Bed, TeddyBear, Blanket.', Room = 'Bedroom'."
    Example 5: Objects = {objects}, Room = """

    inputs = tokenizer(prompt, return_tensors="pt")
    output_ids = model.generate(**inputs, max_new_tokens=30)
    return tokenizer.decode(output_ids[0][-130:], skip_special_tokens=True)

In [46]:
#objects = "Towel, Slippers, Cup, Scales"
#objects = "DogBed, Sofa, Newspaper"
objects = "Sofa, TV, Bookshelf"
pred = classify_room(model, tokenizer, objects)
print(f"Objects: {objects}")
print(f"Predicted room: {pred}")

[transformers] Both `max_new_tokens` (=30) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Objects: Sofa, TV, Bookshelf
Predicted room:  ChoppingBoard, apple.', Room = 'Kitchen'."
    Example 2: Objects = 'Toilet, Shower, Toothbrush.', Room = 'Bathroom'."
    Example 3: Objects = 'Sofa, TV, Bookshelf.', Room = 'Living_Room'."
    Example 4: Objects = 'Bed, TeddyBear, Blanket.', Room = 'Bedroom'."
    Example 5: Objects = Sofa, TV, Bookshelf, Room =  'Living_Room'."
    Example 6: Objects = 'Microwave, ChoppingBoard, apple.', Room = 'Kitchen'."
   


In [47]:
objects = "DogBed, Sofa, Newspaper"
pred = classify_room(model, tokenizer, objects)
print(f"Objects: {objects}")
print(f"Predicted room: {pred}")

[transformers] Both `max_new_tokens` (=30) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Objects: DogBed, Sofa, Newspaper
Predicted room:  ChoppingBoard, apple.', Room = 'Kitchen'."
    Example 2: Objects = 'Toilet, Shower, Toothbrush.', Room = 'Bathroom'."
    Example 3: Objects = 'Sofa, TV, Bookshelf.', Room = 'Living_Room'."
    Example 4: Objects = 'Bed, TeddyBear, Blanket.', Room = 'Bedroom'."
    Example 5: Objects = DogBed, Sofa, Newspaper, Room =  'Bedroom'."
    Example 6: Objects = 'Sofa, TV, Bookshelf, CoffeeMachine, Microwave, ChoppingBoard


In [48]:
objects = "Towel, Slippers, Cup, Scales"
#objects = "DogBed, Sofa, Newspaper"
#objects = "Sofa, TV, Bookshelf."
pred = classify_room(model, tokenizer, objects)
print(f"Objects: {objects}")
print(f"Predicted room: {pred}")

[transformers] Both `max_new_tokens` (=30) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Objects: Towel, Slippers, Cup, Scales
Predicted room:  apple.', Room = 'Kitchen'."
    Example 2: Objects = 'Toilet, Shower, Toothbrush.', Room = 'Bathroom'."
    Example 3: Objects = 'Sofa, TV, Bookshelf.', Room = 'Living_Room'."
    Example 4: Objects = 'Bed, TeddyBear, Blanket.', Room = 'Bedroom'."
    Example 5: Objects = Towel, Slippers, Cup, Scales, Room =  'Kitchen'."
    Example 6: Objects = 'Toilet, Shower, Toothbrush.', Room = 'Kitchen'."
    Example 7


In [1]:
def construct_classifier_question(query_words):
    template = """
    I observe the following objects while exploring a room:
    {0}

    What kind of room is this?

    1. Living room
    2. Kitchen
    3. Bedroom
    4. Bathroom

    You should always provide justification
    """
    #You should always provide justification and confidence estimate of your guess
    question = template.format(query_words)

    return question

In [6]:
def ask_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    output_ids = model.generate(**inputs, max_new_tokens=30)
    return tokenizer.decode(output_ids[0][-130:], skip_special_tokens=True)

In [11]:
#objects = "DogBed, Sofa, Newspaper"
#objects = "Sofa, TV, Bookshelf"
objects = "Towel, Slippers, Cup, Scales"

prompt = construct_classifier_question(objects)
#print(prompt)
rsp = ask_llm(prompt)
print(rsp)
#pred = classify_room(model, tokenizer, objects)
#print(f"Objects: {objects}")
#print(f"Predicted room: {pred}")


    I observe the following objects while exploring a room:
    Towel, Slippers, Cup, Scales

    What kind of room is this?

    1. Living room
    2. Kitchen
    3. Bedroom
    4. Bathroom

    You should always provide justification
    



    [The answer is 2. Kitchen]

    [Justification: The room has a cup, which is a common item found in a


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
#prompt = "Give me a short introduction to large language model."


#objects = "DogBed, Sofa, Newspaper"
#objects = "Sofa, TV, Bookshelf"
objects = "Towel, Slippers, Cup, Scales"

prompt = construct_classifier_question(objects)


messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

thinking content: <think>
Okay, let's try to figure out what kind of room these objects are. The objects mentioned are a towel, slippers, cup, and scales. 

First, I need to think about the typical objects in different rooms. A living room usually has furniture like chairs, sofas, and maybe some decorations. The towel and slippers are typically for sleeping, so maybe a bedroom. The cup could be in a kitchen, but the scales might be in a kitchen or bathroom. 

Wait, scales are used for measuring, so maybe they're in a kitchen for cooking. But the cup is also in a kitchen. So maybe the kitchen is the answer? But then why is the bathroom mentioned? The bathroom has water and scales. Hmm, but scales are used in kitchens. So maybe the answer is kitchen. But let me check again. 

Alternatively, the towel and slippers are for sleeping, so bedroom. The cup would be in a kitchen, and scales in a kitchen. So maybe bedroom and kitchen. But the options are only 1 to 4. The question says "What kind